# EDGAR Financial Sentiment — Part 6: Synthetic Data Generation

**The Track B → Track A bridge.** In Part 5 you used Mistral to *classify*. Here you use the same model to *manufacture labeled training data*, add it to a small real training set, and **measure** whether it helps a downstream classifier.

**What you'll practice (the core concept):** prompting an LLM to **generate labeled examples** (`build_generation_prompt`) and **parsing** the generated text into clean `(sentence, label)` pairs (`parse_generated`) — those are your blanks. Everything else (the generation driver and the downstream bert-base classifier that measures the effect) is provided.

> **Run in Google Colab with a T4 GPU.** Uses the gated Mistral model (Part 4 login) for generation, then a small bert-base classifier to measure the payoff.

## 0. Why generate data? The labeling bottleneck

Fine-tuning (Parts 2–4) needs **labeled** data, and the labels are the expensive part. Financial PhraseBank exists only because annotators hand-labeled thousands of sentences. What if you *don't* have that — a new domain, a new task, no budget for labeling?

**Track B's trick:** ask an LLM to write the examples *for* you. Prompt it for "10 short financial sentences with negative sentiment," and every sentence comes **pre-labeled** — you asked for negative, so the label is negative. Add a few hundred of those to a small real training set and you've expanded your labels almost for free. The same model that *classified* in Part 5 now *generates training data* here.

**The honest question this notebook measures:** *does it actually help?* Synthetic data should help most when real labels are **scarce**, so we simulate that — a tiny real seed set (a few dozen examples), augmented with LLM-generated sentences — and we measure the accuracy change on the **real** test set. Sometimes it lifts accuracy; sometimes synthetic noise (templated or mislabeled sentences) hurts. We find out by **measuring**, not assuming.

**The experiment:**
1. Take a *small* real labeled seed set (simulating scarcity).
2. Prompt Mistral to generate labeled synthetic sentences. *(your code)*
3. Train a bert-base classifier on **real-only** vs. **real + synthetic**.
4. Compare test accuracy — the delta is what the synthetic data bought.

## 1. Setup

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # reduce fragmentation (Part 4 lesson)

import random, re, gc, io, zipfile, requests
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from datasets import Dataset as HFDataset, ClassLabel

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

### Hugging Face login (Mistral is gated)
Same as Parts 4–5: accept the license, make a token, log in.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & seeds

In [ ]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID           = 'mistralai/Mistral-7B-Instruct-v0.3'
LABELS             = ['negative', 'neutral', 'positive']

SEED_PER_CLASS     = 20    # tiny REAL training set per class (simulates scarce labels: 60 total)
SYNTH_PER_CLASS    = 60    # synthetic sentences to generate per class
SENTENCES_PER_CALL = 12    # how many to ask for in a single generation call
GEN_TEMPERATURE    = 0.9   # sampling temperature -> diversity (greedy would repeat itself)

## 3. Real data — a *small* seed set + the real test set

Same PhraseBank + split as before, but we deliberately keep only `SEED_PER_CLASS` real examples per class for training (the scarcity we're simulating). The test set stays full and **real** — that's what we measure on.

In [ ]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')
_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)
pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=LABELS))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']

# tiny REAL seed set: SEED_PER_CLASS examples per class
seed_examples = []
for _cls in range(3):
    _exs = [e for e in pb_train if e['label'] == _cls][:SEED_PER_CLASS]
    seed_examples.extend((e['sentence'], _cls) for e in _exs)
random.shuffle(seed_examples)
print('Real seed examples:', len(seed_examples), '| Real test examples:', len(pb_test))

## 4. Load Mistral-7B (4-bit) for generation  *(provided)*

Loaded as a causal LM, same as Part 5. Note `generate` uses **sampling** (`do_sample=True`, temperature) — unlike Part 5's greedy decoding — because we want *diverse* sentences, not the same one every call.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

In [ ]:
@torch.no_grad()
def generate(prompt_text, max_new_tokens=320, temperature=GEN_TEMPERATURE, top_p=0.95):
    """Sample a reply from the model (sampling -> diverse generations)."""
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=top_p, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

## 5. Write the generation prompt  &larr; **your code**

This is the heart of synthetic-data generation. Write `build_generation_prompt(label_name, n)` that asks the model for `n` short financial sentences expressing a **given** sentiment. Good generation prompts:
- state the **sentiment** clearly (the label you'll attach to every sentence it returns);
- ask for **variety** (different companies, sectors, wording) so the data isn't templated;
- demand a **clean format** you can parse — a numbered list, one sentence per line, no commentary.

The format you ask for here determines how easy `parse_generated` is next — design them together.

In [ ]:
def build_generation_prompt(label_name, n):
    """Ask the model for n short financial sentences with the given sentiment."""
    ### YOUR CODE HERE
    return (
        f'Write {n} short, realistic one-sentence statements that could appear in financial news about a '
        f'company, each clearly expressing {label_name.upper()} sentiment toward the company. '
        f'Vary the companies, sectors, and phrasing. '
        f'Output ONLY a numbered list (1., 2., ...), one sentence per line, with no extra commentary.'
    )
    ### END YOUR CODE

### See a generation prompt + a real reply
Run after filling in `build_generation_prompt`. Look at the raw reply — its format is what your parser must handle.

In [ ]:
_demo = build_generation_prompt('negative', 5)
print('=== PROMPT ===')
print(_demo)
print('\n=== MODEL REPLY ===')
print(generate(_demo))

## 6. Parse the generated sentences  &larr; **your code**

`parse_generated(text)` turns the model's reply (a numbered list) into a clean **list of sentence strings**. For each line: keep only lines that look like `1. ...` / `2) ...`, strip the numbering and surrounding quotes, and drop anything too short to be a real sentence. (A regex like `^\d+[.)]\s*(.+)$` matches a leading number.)

In [ ]:
def parse_generated(text):
    """Pull individual sentences out of the model's numbered-list reply."""
    ### YOUR CODE HERE
    sentences = []
    for line in text.splitlines():
        line = line.strip()
        m = re.match(r'^\d+[.)]\s*(.+)$', line)     # lines like '1. ...' or '1) ...'
        if m:
            s = m.group(1).strip().strip('"')
            if len(s) > 10:                            # drop fragments
                sentences.append(s)
    return sentences
    ### END YOUR CODE

### Test your parser

In [ ]:
_sample = '1. Acme Corp posted a steep quarterly loss.\n2) Beta Inc cut its full-year guidance.\nHere you go:\n3. Gamma Ltd missed revenue estimates.'
print(parse_generated(_sample))   # expect 3 clean sentences, the 'Here you go:' line dropped

## 7. Generate the synthetic dataset  *(provided driver — uses your two functions)*

Loops over the three classes, calls your prompt + parser until it has `SYNTH_PER_CLASS` sentences each, and labels every sentence with the class it asked for. This is the slow cell (many generations) — a few minutes.

In [ ]:
def generate_synthetic(per_class, per_call=SENTENCES_PER_CALL, max_calls=25):
    synth = []
    for cls, name in enumerate(LABELS):
        collected, calls = [], 0
        while len(collected) < per_class and calls < max_calls:
            collected.extend(parse_generated(generate(build_generation_prompt(name, per_call))))
            calls += 1
        synth.extend((s, cls) for s in collected[:per_class])
        print(f'  {name}: collected {len(collected)} -> kept {min(len(collected), per_class)}')
    return synth

print('Generating synthetic data (this takes a few minutes)...')
SYNTH = generate_synthetic(SYNTH_PER_CLASS)
print('Total synthetic (raw):', len(SYNTH))

## 8. Inspect quality — dedup, balance, eyeball  *(instrument)*

Synthetic data is only as good as the generator. Before trusting it: remove any sentence that duplicates a real seed/test sentence (leakage) or repeats another synthetic one, check the class balance, and **read a few of them**.

In [ ]:
real_texts = {s.lower().strip() for s, _ in seed_examples}
real_texts |= {e['sentence'].lower().strip() for e in pb_test}

seen, synth_clean = set(), []
for s, c in SYNTH:
    k = s.lower().strip()
    if k in real_texts or k in seen:
        continue
    seen.add(k); synth_clean.append((s, c))

print(f'synthetic: {len(SYNTH)} raw -> {len(synth_clean)} after dedup')
print('class balance:', dict(Counter(LABELS[c] for _, c in synth_clean)))
print('\nA few generated examples:')
for name in LABELS:
    for s, c in synth_clean:
        if LABELS[c] == name:
            print(f'  [{name}] {s}')
            break

#### ✅ Reflect — is the synthetic data any good?
Read your examples critically. LLM-generated financial sentences are usually **fluent and on-topic**, but watch for the failure modes that make synthetic data *hurt*: **templated sameness** (every sentence is "Company X did Y"), **label drift** (a sentence you asked to be `neutral` actually reads negative — neutral is the hardest to generate cleanly), and **narrow vocabulary** vs. the messy variety of real filings. The dedup count also matters: if lots were dropped as duplicates, your generator wasn't diverse enough (raise temperature, vary the prompt). Quality here caps how much the augmentation can help below.

## 9. Free Mistral, set up the downstream classifier  *(provided)*

We measure the effect with a small, fast **bert-base `[CLS]` classifier** (Part 3) — not the 7B — so we free Mistral from the GPU first. `train_and_eval` fine-tunes a fresh classifier on whatever examples you pass and returns test accuracy on the **real** test set.

In [ ]:
del model
gc.collect(); torch.cuda.empty_cache()
print(f'Freed Mistral. GPU allocated now: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
def train_and_eval(train_examples, epochs=4, lr=2e-5, bs=16, seed=10):
    """Fine-tune a fresh bert-base [CLS] classifier on train_examples [(sentence,label), ...],
    evaluate on the REAL PhraseBank test set, return test accuracy %."""
    torch.manual_seed(seed)
    btok = AutoTokenizer.from_pretrained('bert-base-cased')
    enc = AutoModel.from_pretrained('bert-base-cased').to(device)

    class _DS(Dataset):
        def __init__(self, examples):
            self.rows = []
            for sent, lab in examples:
                e = btok(sent, max_length=64, truncation=True, padding='max_length', return_tensors='pt')
                self.rows.append({'input_ids': e['input_ids'].squeeze(0),
                                  'attention_mask': e['attention_mask'].squeeze(0),
                                  'label': torch.tensor(lab, dtype=torch.int64)})
        def __len__(self): return len(self.rows)
        def __getitem__(self, i): return self.rows[i]

    class _Clf(nn.Module):
        def __init__(self, encoder):
            super().__init__(); self.encoder = encoder
            self.head = nn.Linear(encoder.config.hidden_size, 3)
        def forward(self, b):
            out = self.encoder(input_ids=b['input_ids'], attention_mask=b['attention_mask'])
            return self.head(out.last_hidden_state[:, 0, :])

    test_examples = [(e['sentence'], e['label']) for e in pb_test]
    tr = DataLoader(_DS(train_examples), batch_size=bs, shuffle=True)
    te = DataLoader(_DS(test_examples), batch_size=32, shuffle=False)
    clf = _Clf(enc).to(device); opt = torch.optim.AdamW(clf.parameters(), lr=lr); lf = nn.CrossEntropyLoss()

    clf.train()
    for _ in range(epochs):
        for b in tr:
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad(); lf(clf(b), b['label']).backward(); opt.step()

    clf.eval(); correct = total = 0
    with torch.no_grad():
        for b in te:
            b = {k: v.to(device) for k, v in b.items()}
            correct += (clf(b).argmax(-1) == b['label']).sum().item(); total += b['label'].size(0)
    acc = 100 * correct / total
    del clf, enc; gc.collect(); torch.cuda.empty_cache()
    return acc

## 10. Measure the effect — real-only vs. real + synthetic

#### ✅ A reasonable prediction
With only ~60 real examples, bert-base will likely land somewhere in the **mid-80s to low-90s** — already strong, because PhraseBank-all-agree is an easy, clean task. Adding decent synthetic data usually gives a **small positive lift** (a few points) by covering cases the tiny seed set missed. But don't expect miracles, and a *negative* result is entirely possible if the synthetic labels are noisy — that's a real finding about generator quality, not a bug.

In [ ]:
print(f'Training on REAL seed only ({len(seed_examples)} examples)...')
acc_real = train_and_eval(seed_examples)
print(f'  -> {acc_real:.1f}%')
print(f'Training on REAL + SYNTHETIC ({len(seed_examples) + len(synth_clean)} examples)...')
acc_aug = train_and_eval(seed_examples + synth_clean)
print(f'  -> {acc_aug:.1f}%')

print('\n================ RESULT ================')
print(f'Real only:        {acc_real:.1f}%')
print(f'Real + synthetic: {acc_aug:.1f}%')
print(f'Synthetic lift:   {acc_aug - acc_real:+.1f} pts')

## 11. Reflect — when does synthetic data help?

#### ✅ The takeaways
- **Synthetic data is a *scarcity* tool.** It helps most when real labels are few (like our 60-example seed). Re-run with the **full** `pb_train` as the seed and the lift usually shrinks to nothing — there's no gap left to fill. That's why we simulated scarcity.
- **It's capped by generator quality.** The LLM can only produce labels as good as its own understanding; mislabeled or templated sentences add **noise**, which is why a careless prompt can make accuracy *drop*. Garbage in, garbage out — your `build_generation_prompt` is doing real work.
- **It's the cheapest way to attack the labeling bottleneck**, and it composes with everything else: generate data here, then fine-tune (Parts 2–4) or even distill a Part-5 prompt into a trained model. Track B feeding Track A.

**Things to try:** raise `SYNTH_PER_CLASS`; add 2–3 *real* examples into the generation prompt (few-shot generation → more realistic output); raise/lower temperature and watch the dedup count; re-run with the full `pb_train` seed to see the lift vanish.

## 12. Recap & next

You turned a *generator* into a *data factory*: a prompt that emits labeled sentences, a parser to clean them, a quality/dedup pass, and a measured before/after on a real test set. The lesson isn't "synthetic data is good" — it's that it's a **scarcity tool whose value you must measure**, capped by how good your generation prompt is.

**Next:** Part 7 — **structured extraction**: prompt the LLM to emit **JSON** (sentiment + guidance direction + key figures) from real press-release text, with chain-of-thought rationales — the on-ramp to real 8-K data.